In [1]:
"""
Evaluatie — 6 Modellen
========================
Evalueert de volgende modellen op de testset (2024/25 en 2025/26):

  1. Naive baseline
  2. Dixon-Coles (goals)
  3. Dixon-Coles (xG)
  4. EKF basis                    <- kalman_basis.csv
  5. EKF + asymmetrie             <- kalman_asymmetrie.csv
  6. EKF + asymmetrie + lineup    <- kalman_asymmetrie_lineup.csv

Dixon-Coles wordt vóór elke testset wedstrijd opnieuw geschat op alle
beschikbare data tot dat moment (per-wedstrijd rolling), met warm-start
voor snelheid. Dit is methodologisch consistent met de EKF die ook na
elke wedstrijd updatet.

Metrics: RPS, Brier Score, Accuracy
Output:  evaluation_results.csv

Input vereist:
  - understat_xg.csv
  - kalman_basis.csv
  - kalman_asymmetrie.csv
  - kalman_asymmetrie_lineup.csv
"""

import os
import warnings
import numpy as np
import pandas as pd
from scipy.stats import poisson
from scipy.optimize import minimize

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# PADEN
# ══════════════════════════════════════════════════════════════════════════════
BASE_DIR       = r"C:\Users\semwi\FPL-Core-Insights_Thesis\data"
UNDERSTAT_PATH = os.path.join(BASE_DIR, "understat_xg.csv")
OUTPUT_PATH    = os.path.join(BASE_DIR, "evaluation_results.csv")

KALMAN_PATHS = {
    "EKF basis":                 os.path.join(BASE_DIR, "kalman_basis.csv"),
    "EKF + asymmetrie":          os.path.join(BASE_DIR, "kalman_asymmetrie.csv"),
    "EKF + asymmetrie + lineup": os.path.join(BASE_DIR, "kalman_asymmetrie_lineup.csv"),
}

UD_TEAM_MAP = {
    "Tottenham":     "Tottenham Hotspur",
    "Newcastle":     "Newcastle United",
    "Wolves":        "Wolverhampton Wanderers",
    "Wolverhampton": "Wolverhampton Wanderers",
    "Brighton":      "Brighton & Hove Albion",
    "West Ham":      "West Ham United",
    "Leicester":     "Leicester City",
    "Ipswich":       "Ipswich Town",
    "Luton":         "Luton Town",
    "Norwich":       "Norwich City",
    "Leeds":         "Leeds United",
}

EXCLUDE_SEASONS = {"2014/2015", "2015/2016"}
TRAIN_SEASONS   = {f"{y}/{y+1}" for y in range(2016, 2024)}
TEST_SEASONS    = {"2024/2025", "2025/2026"}

RHO       = -0.13
MAX_GOALS = 12


# ══════════════════════════════════════════════════════════════════════════════
# HULPFUNCTIES
# ══════════════════════════════════════════════════════════════════════════════
def outcome_index(hg, ag):
    if hg > ag:    return 0   # home win
    elif hg == ag: return 1   # draw
    else:          return 2   # away win

def rps_score(p_home, p_draw, p_away, outcome):
    probs = np.array([p_home, p_draw, p_away])
    obs   = np.zeros(3)
    obs[outcome] = 1.0
    return np.mean((np.cumsum(probs[:2]) - np.cumsum(obs[:2])) ** 2)

def brier_score(p_home, p_draw, p_away, outcome):
    probs = np.array([p_home, p_draw, p_away])
    obs   = np.zeros(3)
    obs[outcome] = 1.0
    return np.sum((probs - obs) ** 2)

def dixon_coles_tau(i, j, lh, la, rho):
    if   i == 0 and j == 0: return 1 - lh * la * rho
    elif i == 1 and j == 0: return 1 + la * rho
    elif i == 0 and j == 1: return 1 + lh * rho
    elif i == 1 and j == 1: return 1 - rho
    return 1.0

def lambda_to_probs(lh, la, rho=RHO, max_goals=MAX_GOALS):
    ph = pd_ = pa = 0.0
    for i in range(max_goals + 1):
        for j in range(max_goals + 1):
            p = (poisson.pmf(i, max(lh, 1e-6)) *
                 poisson.pmf(j, max(la, 1e-6)) *
                 dixon_coles_tau(i, j, lh, la, rho))
            if   i > j:  ph  += p
            elif i == j: pd_ += p
            else:        pa  += p
    total = ph + pd_ + pa
    if total < 1e-9:
        return 1/3, 1/3, 1/3
    return ph / total, pd_ / total, pa / total

def eval_row(model_name, p_home, p_draw, p_away, outcome,
             date=None, home_team=None, away_team=None):
    return {
        "model":      model_name,
        "date":       date,
        "home_team":  home_team,
        "away_team":  away_team,
        "match_id":   f"{date}_{home_team}_{away_team}" if date else None,
        "rps":        rps_score(p_home, p_draw, p_away, outcome),
        "brier":      brier_score(p_home, p_draw, p_away, outcome),
        "correct":    int(np.argmax([p_home, p_draw, p_away]) == outcome),
    }


# ══════════════════════════════════════════════════════════════════════════════
# DATA LADEN
# ══════════════════════════════════════════════════════════════════════════════
print("Data laden...")
df = pd.read_csv(UNDERSTAT_PATH)
df["home_team"] = df["home_team"].replace(UD_TEAM_MAP)
df["away_team"] = df["away_team"].replace(UD_TEAM_MAP)
df["date"]      = pd.to_datetime(df["date"], errors="coerce")
df = df[~df["season"].isin(EXCLUDE_SEASONS)].copy()
df = df.sort_values("date").reset_index(drop=True)
df_xg = df[df["home_xg"].notna() & df["away_xg"].notna()].copy().reset_index(drop=True)

train_df = df_xg[df_xg["season"].isin(TRAIN_SEASONS)].copy()
test_df  = df_xg[df_xg["season"].isin(TEST_SEASONS)].copy().reset_index(drop=True)
print(f"  Trainingsset: {len(train_df)} wedstrijden")
print(f"  Testset:      {len(test_df)} wedstrijden")

all_rows = []


# ══════════════════════════════════════════════════════════════════════════════
# MODEL 1 — NAIVE BASELINE
# ══════════════════════════════════════════════════════════════════════════════
print("\n[1/6] Naive baseline...")
outcomes_train = train_df.apply(
    lambda r: outcome_index(r["home_goals"], r["away_goals"]), axis=1)
p_h = (outcomes_train == 0).mean()
p_d = (outcomes_train == 1).mean()
p_a = (outcomes_train == 2).mean()

for _, row in test_df.iterrows():
    if pd.isna(row["home_goals"]) or pd.isna(row["away_goals"]):
        continue
    oc = outcome_index(int(row["home_goals"]), int(row["away_goals"]))
    all_rows.append(eval_row("Naive", p_h, p_d, p_a, oc,
                             date=str(row["date"].date()),
                             home_team=row["home_team"],
                             away_team=row["away_team"]))

print(f"  -> {sum(1 for r in all_rows if r['model'] == 'Naive')} wedstrijden")


# ══════════════════════════════════════════════════════════════════════════════
# DIXON-COLES SCHATTER
# ══════════════════════════════════════════════════════════════════════════════
def fit_dixon_coles(matches_df, obs_h, obs_a, xi=0.00325, x0_init=None):
    """
    Schat Dixon-Coles parameters via gewogen maximum likelihood.
    x0_init: warm-start beginpunt (vorige oplossing), versnelt convergentie.
    Geeft ook de oplossing terug als x0 voor de volgende aanroep.
    """
    teams = sorted(set(matches_df["home_team"]) | set(matches_df["away_team"]))
    t_idx = {t: i for i, t in enumerate(teams)}
    n     = len(teams)

    t_now   = matches_df["date"].max()
    t_diff  = (t_now - matches_df["date"]).dt.days.values.astype(float)
    weights = np.exp(-xi * t_diff)

    obs_h_vals = matches_df[obs_h].values.astype(float)
    obs_a_vals = matches_df[obs_a].values.astype(float)
    hi = matches_df["home_team"].map(t_idx).values
    ai = matches_df["away_team"].map(t_idx).values

    def neg_ll(params):
        att  = params[:n]
        deff = params[n:2*n]
        hav  = params[2*n]
        lh   = np.exp(np.clip(att[hi] + deff[ai] + hav, -5, 5))
        la   = np.exp(np.clip(att[ai] + deff[hi],       -5, 5))
        ll   = (weights * (
            np.log(np.maximum(poisson.pmf(np.round(obs_h_vals).astype(int), lh), 1e-300)) +
            np.log(np.maximum(poisson.pmf(np.round(obs_a_vals).astype(int), la), 1e-300))
        )).sum()
        return -ll

    # Warm-start: hergebruik vorige oplossing als die beschikbaar is en
    # de teamdimensie overeenkomt, anders begin opnieuw
    if x0_init is not None and len(x0_init) == 2*n + 1:
        x0 = x0_init.copy()
    else:
        x0      = np.zeros(2*n + 1)
        x0[2*n] = 0.3

    res = minimize(neg_ll, x0, method="L-BFGS-B",
                   options={"maxiter": 2000, "ftol": 1e-9})

    att_d = {t: res.x[t_idx[t]]     for t in teams}
    def_d = {t: res.x[n + t_idx[t]] for t in teams}
    hav   = float(res.x[2*n])
    return att_d, def_d, hav, teams, res.x

def predict_dc(att_d, def_d, hav, home_team, away_team):
    lh = np.exp(att_d.get(home_team, 0) + def_d.get(away_team, 0) + hav)
    la = np.exp(att_d.get(away_team, 0) + def_d.get(home_team, 0))
    return lambda_to_probs(lh, la)


# ══════════════════════════════════════════════════════════════════════════════
# MODELLEN 2 & 3 — DIXON-COLES (GOALS en XG)
# Schat vóór elke testset wedstrijd opnieuw op alle data tot dat moment.
# Warm-start hergebruikt de vorige oplossing als beginpunt voor snelheid.
# ══════════════════════════════════════════════════════════════════════════════
def run_dc_per_match(df_all, obs_h, obs_a, model_name):
    print(f"\n[DC] {model_name} — schatten per wedstrijd...")
    df_local  = df_all.sort_values("date").reset_index(drop=True)
    test_rows = df_local[df_local["season"].isin(TEST_SEASONS)].reset_index(drop=True)
    total     = len(test_rows)

    rows    = []
    x0_warm = None

    for i, (_, row) in enumerate(test_rows.iterrows()):
        if pd.isna(row["home_goals"]) or pd.isna(row["away_goals"]):
            continue

        # Alle wedstrijden gespeeld VOOR deze wedstrijd
        mask = (
            (df_local["date"] < row["date"]) &
            (df_local[obs_h].notna()) &
            (df_local[obs_a].notna())
        )
        train_window = df_local[mask].copy()
        if len(train_window) < 100:
            continue

        try:
            att_d, def_d, hav, _, x0_warm = fit_dixon_coles(
                train_window, obs_h, obs_a, x0_init=x0_warm)
        except Exception as e:
            print(f"  Schatten mislukt voor wedstrijd {i}: {e}")
            x0_warm = None
            continue

        h = row["home_team"]
        a = row["away_team"]
        if h not in att_d or a not in att_d:
            continue

        ph, pd_, pa = predict_dc(att_d, def_d, hav, h, a)
        oc = outcome_index(int(row["home_goals"]), int(row["away_goals"]))
        rows.append(eval_row(model_name, ph, pd_, pa, oc,
                             date=str(row["date"].date()) if hasattr(row["date"], "date") else str(row["date"]),
                             home_team=h, away_team=a))

        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{total} verwerkt...")

    print(f"  -> {len(rows)} wedstrijden geëvalueerd")
    return rows


print("\n[2/6] Dixon-Coles (goals)...")
all_rows += run_dc_per_match(
    df[df["home_goals"].notna() & df["away_goals"].notna()].copy(),
    obs_h="home_goals", obs_a="away_goals",
    model_name="Dixon-Coles (goals)"
)

print("\n[3/6] Dixon-Coles (xG)...")
all_rows += run_dc_per_match(
    df_xg.copy(),
    obs_h="home_xg", obs_a="away_xg",
    model_name="Dixon-Coles (xG)"
)


# ══════════════════════════════════════════════════════════════════════════════
# MODELLEN 4, 5, 6 — EKF VARIANTEN (inladen uit CSV)
# ══════════════════════════════════════════════════════════════════════════════
for model_name, path in KALMAN_PATHS.items():
    print(f"\n[EKF] {model_name}...")
    kdf  = pd.read_csv(path)
    kdf  = kdf[kdf["season"].isin(TEST_SEASONS)].copy().reset_index(drop=True)
    rows = []

    for _, row in kdf.iterrows():
        if pd.isna(row["home_goals"]) or pd.isna(row["away_goals"]):
            continue
        ph, pd_, pa = lambda_to_probs(
            row["kalman_xg_pred_home"],
            row["kalman_xg_pred_away"]
        )
        oc = outcome_index(int(row["home_goals"]), int(row["away_goals"]))
        rows.append(eval_row(model_name, ph, pd_, pa, oc,
                             date=str(row["date"]),
                             home_team=row["home_team"],
                             away_team=row["away_team"]))

    print(f"  -> {len(rows)} wedstrijden geëvalueerd")
    all_rows += rows


# ══════════════════════════════════════════════════════════════════════════════
# SAMENVATTINGSTABEL
# ══════════════════════════════════════════════════════════════════════════════
results_df = pd.DataFrame(all_rows)

summary = (results_df
           .groupby("model")
           .agg(
               N        =("rps",     "count"),
               RPS      =("rps",     "mean"),
               Brier    =("brier",   "mean"),
               Accuracy =("correct", "mean"),
           )
           .reset_index())

order = [
    "Naive",
    "Dixon-Coles (goals)",
    "Dixon-Coles (xG)",
    "EKF basis",
    "EKF + asymmetrie",
    "EKF + asymmetrie + lineup",
]
summary["_order"] = summary["model"].map({m: i for i, m in enumerate(order)})
summary = (summary
           .sort_values("_order")
           .drop(columns="_order")
           .round({"RPS": 4, "Brier": 4, "Accuracy": 4}))

print("\n" + "="*65)
print("EVALUATIE RESULTATEN — testset 2024/25 en 2025/26")
print("="*65)
print(summary.to_string(index=False))

summary.to_csv(OUTPUT_PATH, index=False)
print(f"\nOpgeslagen: {OUTPUT_PATH}")

# ══════════════════════════════════════════════════════════════════════════════
# PER-WEDSTRIJD SCORES OPSLAAN (voor significantietests)
# ══════════════════════════════════════════════════════════════════════════════
per_match_path = os.path.join(BASE_DIR, "evaluation_per_match.csv")
results_df.to_csv(per_match_path, index=False)
print(f"Per-wedstrijd scores opgeslagen: {per_match_path}")

Data laden...
  Trainingsset: 3040 wedstrijden
  Testset:      739 wedstrijden

[1/6] Naive baseline...
  -> 739 wedstrijden

[2/6] Dixon-Coles (goals)...

[DC] Dixon-Coles (goals) — schatten per wedstrijd...
  50/739 verwerkt...
  100/739 verwerkt...
  150/739 verwerkt...
  200/739 verwerkt...
  250/739 verwerkt...
  300/739 verwerkt...
  350/739 verwerkt...
  400/739 verwerkt...
  450/739 verwerkt...
  500/739 verwerkt...
  550/739 verwerkt...
  600/739 verwerkt...
  650/739 verwerkt...
  700/739 verwerkt...
  -> 738 wedstrijden geëvalueerd

[3/6] Dixon-Coles (xG)...

[DC] Dixon-Coles (xG) — schatten per wedstrijd...
  50/739 verwerkt...
  100/739 verwerkt...
  150/739 verwerkt...
  200/739 verwerkt...
  250/739 verwerkt...
  300/739 verwerkt...
  350/739 verwerkt...
  400/739 verwerkt...
  450/739 verwerkt...
  500/739 verwerkt...
  550/739 verwerkt...
  600/739 verwerkt...
  650/739 verwerkt...
  700/739 verwerkt...
  -> 738 wedstrijden geëvalueerd

[EKF] EKF basis...
  -> 739 weds

In [2]:
"""
Significantietests — Paarsgewijze Modelcomparison
===================================================
Produceert een nette tabel voor in een academisch paper.
Alle 15 unieke paren worden getest (6 modellen -> 6*5/2 = 15).

Tests:
  - Wilcoxon signed-rank test (primair)
  - Diebold-Mariano test (aanvullend)

Brier Score gedeeld door K=3 (bereik 0–2/3), consistent met literatuur.
Accuracy = fractie matches waarbij argmax(p) == werkelijke uitkomst.

Input:  evaluation_per_match.csv
Output: significance_tests.csv
"""

import os
import itertools
import warnings
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon, norm

warnings.filterwarnings("ignore")

BASE_DIR       = r"C:\Users\semwi\FPL-Core-Insights_Thesis\data"
PER_MATCH_PATH = os.path.join(BASE_DIR, "evaluation_per_match.csv")
OUTPUT_PATH    = os.path.join(BASE_DIR, "significance_tests.csv")

MODEL_ORDER = [
    "Naive",
    "Dixon-Coles (goals)",
    "Dixon-Coles (xG)",
    "EKF basis",
    "EKF + asymmetrie",
    "EKF + asymmetrie + lineup",
]

K = 3  # aantal uitkomsten (home win, draw, away win)


# ══════════════════════════════════════════════════════════════════════════════
# HULPFUNCTIES
# ══════════════════════════════════════════════════════════════════════════════
def diebold_mariano(losses_a, losses_b):
    d      = np.array(losses_a) - np.array(losses_b)
    n      = len(d)
    d_mean = np.mean(d)
    nw_var = np.var(d, ddof=1)
    if nw_var <= 0:
        return np.nan, np.nan
    dm_stat = d_mean / np.sqrt(nw_var / n)
    p_val   = 2 * norm.cdf(-abs(dm_stat))
    return float(dm_stat), float(p_val)

def sig_stars(p):
    if pd.isna(p):  return ""
    if p < 0.001:   return "***"
    elif p < 0.01:  return "**"
    elif p < 0.05:  return "*"
    else:           return ""


# ══════════════════════════════════════════════════════════════════════════════
# DATA LADEN
# ══════════════════════════════════════════════════════════════════════════════
df = pd.read_csv(PER_MATCH_PATH)

if "match_id" not in df.columns:
    df["match_id"] = (df["date"].astype(str) + "_" +
                      df["home_team"] + "_" +
                      df["away_team"])

# Brier Score delen door K voor consistentie met literatuur (bereik 0 – 2/3)
# Controleer of de brier kolom al gedeeld is of niet
# We delen door K hier zodat het consistent is met de thesis formule /K
if df["brier"].mean() > 0.4:
    # Waarden zijn nog niet gedeeld door K (bereik ~0-2), deel nu
    print("INFO: Brier Score gedeeld door K=3 voor normalisatie.")
    df["brier"] = df["brier"] / K
else:
    print("INFO: Brier Score lijkt al genormaliseerd, geen aanpassing.")

# Accuracy per match: 1 als argmax(p) == uitkomst, anders 0
# Verwacht kolommen: p_home, p_draw, p_away, result (H/D/A of 0/1/2)
if all(c in df.columns for c in ["p_home", "p_draw", "p_away", "result"]):
    outcome_map = {"H": 0, "D": 1, "A": 2}
    if df["result"].dtype == object:
        df["result_idx"] = df["result"].map(outcome_map)
    else:
        df["result_idx"] = df["result"]
    df["predicted_idx"] = df[["p_home", "p_draw", "p_away"]].values.argmax(axis=1)
    df["correct"] = (df["predicted_idx"] == df["result_idx"]).astype(int)
    has_accuracy = True
else:
    has_accuracy = False
    print("WAARSCHUWING: Kolommen p_home/p_draw/p_away/result niet gevonden. Accuracy overgeslagen.")

model_dfs     = {m: df[df["model"] == m].set_index("match_id") for m in df["model"].unique()}
models_to_use = [m for m in MODEL_ORDER if m in model_dfs]


# ══════════════════════════════════════════════════════════════════════════════
# SAMENVATTING PER MODEL (voor Table 5.1)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("SAMENVATTING PER MODEL (Table 5.1)")
print("=" * 70)
summary_rows = []
for model in models_to_use:
    mdf = model_dfs[model]
    row = {
        "Model": model,
        "N": len(mdf),
        "RPS": round(mdf["rps"].mean(), 4),
        "Brier": round(mdf["brier"].mean(), 4),
    }
    if has_accuracy and "correct" in mdf.columns:
        row["Accuracy"] = round(mdf["correct"].mean(), 4)
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))


# ══════════════════════════════════════════════════════════════════════════════
# ALLE 15 PAARSGEWIJZE TESTS
# ══════════════════════════════════════════════════════════════════════════════
rows = []
for model_a, model_b in itertools.combinations(models_to_use, 2):
    df_a   = model_dfs[model_a]
    df_b   = model_dfs[model_b]
    common = sorted(set(df_a.index) & set(df_b.index))
    if len(common) < 10:
        continue

    rps_a   = df_a.loc[common, "rps"].values
    rps_b   = df_b.loc[common, "rps"].values
    br_a    = df_a.loc[common, "brier"].values
    br_b    = df_b.loc[common, "brier"].values
    n       = len(common)

    try:
        _, p_w_rps   = wilcoxon(rps_a, rps_b, alternative="two-sided")
        _, p_w_brier = wilcoxon(br_a,  br_b,  alternative="two-sided")
    except ValueError:
        p_w_rps = p_w_brier = np.nan

    _, p_dm_rps   = diebold_mariano(rps_a, rps_b)
    _, p_dm_brier = diebold_mariano(br_a,  br_b)

    rows.append({
        "Model A":              model_a,
        "Model B":              model_b,
        "N":                    n,
        "RPS A":                round(rps_a.mean(), 4),
        "RPS B":                round(rps_b.mean(), 4),
        "ΔRPS (A−B)":           round(rps_a.mean() - rps_b.mean(), 4),
        "Wilcoxon p (RPS)":     round(p_w_rps,   4) if not np.isnan(p_w_rps)   else np.nan,
        "Sig (RPS)":            sig_stars(p_w_rps),
        "DM p (RPS)":           round(p_dm_rps,  4) if not np.isnan(p_dm_rps)  else np.nan,
        "Sig DM (RPS)":         sig_stars(p_dm_rps),
        "Brier A":              round(br_a.mean(),  4),
        "Brier B":              round(br_b.mean(),  4),
        "ΔBrier (A−B)":         round(br_a.mean()  - br_b.mean(),  4),
        "Wilcoxon p (Brier)":   round(p_w_brier, 4) if not np.isnan(p_w_brier) else np.nan,
        "Sig (Brier)":          sig_stars(p_w_brier),
        "DM p (Brier)":         round(p_dm_brier,4) if not np.isnan(p_dm_brier) else np.nan,
        "Sig DM (Brier)":       sig_stars(p_dm_brier),
    })

results = pd.DataFrame(rows)

# ══════════════════════════════════════════════════════════════════════════════
# PRINT VOOR PAPER
# ══════════════════════════════════════════════════════════════════════════════
print("\nPAARSGEWIJZE SIGNIFICANTIETESTS")
print("Negatief ΔRPS = Model A beter dan Model B")
print("=" * 95)
print(results[[
    "Model A", "Model B", "N",
    "RPS A", "RPS B", "ΔRPS (A−B)",
    "Wilcoxon p (RPS)", "Sig (RPS)",
    "DM p (RPS)",       "Sig DM (RPS)",
]].to_string(index=False))

print("\n" + "=" * 95)
print(results[[
    "Model A", "Model B", "N",
    "Brier A", "Brier B", "ΔBrier (A−B)",
    "Wilcoxon p (Brier)", "Sig (Brier)",
    "DM p (Brier)",       "Sig DM (Brier)",
]].to_string(index=False))

results.to_csv(OUTPUT_PATH, index=False)
print(f"\nOpgeslagen: {OUTPUT_PATH}")
print("*** p<0.001  ** p<0.01  * p<0.05")

INFO: Brier Score gedeeld door K=3 voor normalisatie.
WAARSCHUWING: Kolommen p_home/p_draw/p_away/result niet gevonden. Accuracy overgeslagen.

SAMENVATTING PER MODEL (Table 5.1)
                    Model   N    RPS  Brier
                    Naive 739 0.2322 0.2191
      Dixon-Coles (goals) 738 0.2056 0.2006
         Dixon-Coles (xG) 738 0.2028 0.1993
                EKF basis 739 0.2034 0.1997
         EKF + asymmetrie 739 0.2033 0.1997
EKF + asymmetrie + lineup 739 0.2032 0.1996

PAARSGEWIJZE SIGNIFICANTIETESTS
Negatief ΔRPS = Model A beter dan Model B
            Model A                   Model B   N  RPS A  RPS B  ΔRPS (A−B)  Wilcoxon p (RPS) Sig (RPS)  DM p (RPS) Sig DM (RPS)
              Naive       Dixon-Coles (goals) 738 0.2321 0.2056      0.0265            0.0000       ***      0.0000          ***
              Naive          Dixon-Coles (xG) 738 0.2321 0.2028      0.0293            0.0000       ***      0.0000          ***
              Naive                 EKF basis 739 0

In [3]:
"""
Betting Simulatie — EKF + asymmetrie vs Naive Baseline vs Betfair Exchange
===========================================================================
Produceert tabellen voor:
  - ROI per drempel (Table 5.x)
  - ROI per uitkomsttype bij 5% drempel (Table 5.x)
  - ROI per kwartaal (Table C.x)
  - ROI per odds range (Table C.x)
  - ROI per team (Table C.x)
  - Random baseline test

Input:  kalman_expected_goals.csv
        odds raw/E0*.csv
"""

import os
import glob
import random
import warnings
import numpy as np
import pandas as pd
from scipy.stats import poisson

warnings.filterwarnings('ignore')

BASE_DIR       = r"C:\Users\semwi\FPL-Core-Insights_Thesis\data"
KALMAN_PATH    = os.path.join(BASE_DIR, "kalman_expected_goals.csv")
ODDS_GLOB      = os.path.join(BASE_DIR, r"odds raw\E0*.csv")

EDGE           = 0.00
COLD_START     = 5
N_RANDOM       = 1000
RANDOM_SEED    = 42
MAX_GOALS      = 12
RHO            = -0.13

ODDS_TEAM_MAP = {
    "Man United":    "Manchester United",
    "Man City":      "Manchester City",
    "Tottenham":     "Tottenham Hotspur",
    "Newcastle":     "Newcastle United",
    "Brighton":      "Brighton & Hove Albion",
    "West Ham":      "West Ham United",
    "Leicester":     "Leicester City",
    "Ipswich":       "Ipswich Town",
    "Luton":         "Luton Town",
    "Wolves":        "Wolverhampton Wanderers",
    "Wolverhampton": "Wolverhampton Wanderers",
    "Nott'm Forest": "Nottingham Forest",
    "Leeds":         "Leeds United",
    "West Brom":     "West Bromwich Albion",
}

# ══════════════════════════════════════════════════════════════════════════════
# STAP 1 — Data laden
# ══════════════════════════════════════════════════════════════════════════════
kalman = pd.read_csv(KALMAN_PATH)
kalman['date'] = pd.to_datetime(kalman['date'])

odds_dfs = []
for path in sorted(glob.glob(ODDS_GLOB)):
    df = pd.read_csv(path, encoding='latin-1')
    df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
    df = df[df['Date'] >= '2020-08-01']
    if len(df) > 0:
        odds_dfs.append(df)

odds = pd.concat(odds_dfs, ignore_index=True)
odds['HomeTeam'] = odds['HomeTeam'].replace(ODDS_TEAM_MAP)
odds['AwayTeam'] = odds['AwayTeam'].replace(ODDS_TEAM_MAP)
odds['_date'] = odds['Date'].dt.date.astype(str)

# ══════════════════════════════════════════════════════════════════════════════
# STAP 2 — Dixon-Coles kansen
# ══════════════════════════════════════════════════════════════════════════════
def dixon_coles_tau(i, j, lh, la, rho):
    if i == 0 and j == 0: return 1 - lh * la * rho
    if i == 1 and j == 0: return 1 + la * rho
    if i == 0 and j == 1: return 1 + lh * rho
    if i == 1 and j == 1: return 1 - rho
    return 1.0

def xg_to_probs(lh, la):
    lh = max(lh, 0.05)
    la = max(la, 0.05)
    ph = pd_ = pa = 0.0
    for i in range(MAX_GOALS + 1):
        for j in range(MAX_GOALS + 1):
            p = (poisson.pmf(i, lh) * poisson.pmf(j, la) *
                 dixon_coles_tau(i, j, lh, la, RHO))
            if i > j:    ph  += p
            elif i == j: pd_ += p
            else:        pa  += p
    total = ph + pd_ + pa
    return ph / total, pd_ / total, pa / total

# ══════════════════════════════════════════════════════════════════════════════
# STAP 3 — EKF kansen berekenen
# ══════════════════════════════════════════════════════════════════════════════
ekf = kalman[kalman['kalman_xg_pred_home'].notna()].copy()
probs = ekf.apply(
    lambda r: xg_to_probs(r['kalman_xg_pred_home'], r['kalman_xg_pred_away']), axis=1)
ekf['ekf_prob_home'] = [p[0] for p in probs]
ekf['ekf_prob_draw'] = [p[1] for p in probs]
ekf['ekf_prob_away'] = [p[2] for p in probs]
ekf['_date'] = ekf['date'].dt.date.astype(str)

# Naive baseline: historische unconditional frequenties op trainingsset
# (worden hieronder berekend na merge)

# ══════════════════════════════════════════════════════════════════════════════
# STAP 4 — Merge EKF + Odds
# ══════════════════════════════════════════════════════════════════════════════
merged = ekf.merge(
    odds[['_date','HomeTeam','AwayTeam',
          'FTHG','FTAG',
          'BFEH','BFED','BFEA']],
    left_on=['_date','home_team','away_team'],
    right_on=['_date','HomeTeam','AwayTeam'],
    how='inner'
).drop(columns=['HomeTeam','AwayTeam','_date'])

merged['home_goals'] = merged['FTHG'].astype(float)
merged['away_goals'] = merged['FTAG'].astype(float)
merged = merged[merged['home_goals'].notna()].copy()
merged['result'] = merged.apply(
    lambda r: 'H' if r['home_goals'] > r['away_goals']
              else ('A' if r['home_goals'] < r['away_goals'] else 'D'), axis=1)
merged = merged.sort_values(['season','date']).copy()

# ══════════════════════════════════════════════════════════════════════════════
# STAP 5 — Cold-start filter (nieuwkomers per seizoen, eerste N wedstrijden)
# ══════════════════════════════════════════════════════════════════════════════
seizoenen   = sorted(merged['season'].unique())
nieuwkomers = set()
for i, s in enumerate(seizoenen):
    if i == 0:
        continue
    cur  = set(merged[merged['season']==s]['home_team']) | set(merged[merged['season']==s]['away_team'])
    prev = set(merged[merged['season']==seizoenen[i-1]]['home_team']) | set(merged[merged['season']==seizoenen[i-1]]['away_team'])
    for t in cur - prev:
        nieuwkomers.add((s, t))

merged['home_match_nr'] = merged.groupby(['season','home_team']).cumcount() + 1
merged['away_match_nr'] = merged.groupby(['season','away_team']).cumcount() + 1

def is_cold(row):
    if (row['season'], row['home_team']) in nieuwkomers and row['home_match_nr'] <= COLD_START:
        return True
    if (row['season'], row['away_team']) in nieuwkomers and row['away_match_nr'] <= COLD_START:
        return True
    return False

cold_mask = merged.apply(is_cold, axis=1)
merged    = merged[~cold_mask].copy()

# ══════════════════════════════════════════════════════════════════════════════
# STAP 6 — Betfair implied kansen + Naive baseline
# ══════════════════════════════════════════════════════════════════════════════
merged['bfe_prob_home'] = 1 / merged['BFEH']
merged['bfe_prob_draw'] = 1 / merged['BFED']
merged['bfe_prob_away'] = 1 / merged['BFEA']
total = merged['bfe_prob_home'] + merged['bfe_prob_draw'] + merged['bfe_prob_away']
merged['bfe_prob_home'] /= total
merged['bfe_prob_draw'] /= total
merged['bfe_prob_away'] /= total

# Naive: unconditional frequenties op de data zelf (out-of-sample: gebruik trainingsset freq)
naive_h = (merged['result'] == 'H').mean()
naive_d = (merged['result'] == 'D').mean()
naive_a = (merged['result'] == 'A').mean()
merged['naive_prob_home'] = naive_h
merged['naive_prob_draw'] = naive_d
merged['naive_prob_away'] = naive_a

merged = merged[merged['BFEH'].notna()].copy()
print(f"Finale dataset: {len(merged)} wedstrijden")

# ══════════════════════════════════════════════════════════════════════════════
# STAP 7 — Simulatie functie
# ══════════════════════════════════════════════════════════════════════════════
def simuleer(df, model_ph, model_pd, model_pa,
             market_ph, market_pd, market_pa,
             odds_h, odds_d, odds_a, edge_threshold):
    """Flat-stake simulatie. Vergelijkt model kansen met markt kansen."""
    bets_log = []
    for _, row in df.iterrows():
        if pd.isna(row[odds_h]):
            continue
        edges = {
            'H': row[model_ph] - row[market_ph],
            'D': row[model_pd] - row[market_pd],
            'A': row[model_pa] - row[market_pa],
        }
        best = max(edges, key=edges.get)
        if edges[best] >= edge_threshold:
            o = row[odds_h if best=='H' else odds_d if best=='D' else odds_a]
            if pd.isna(o) or o <= 1:
                continue
            profit = (o - 1) if row['result'] == best else -1.0
            bets_log.append({
                'date'   : str(row['date'].date()),
                'season' : row['season'],
                'home'   : row['home_team'],
                'away'   : row['away_team'],
                'bet'    : best,
                'result' : row['result'],
                'edge'   : round(edges[best], 4),
                'odds'   : round(o, 2),
                'profit' : round(profit, 2),
                'won'    : row['result'] == best,
                'month'  : row['date'].month,
            })
    return pd.DataFrame(bets_log)


def roi(df):
    if len(df) == 0: return 0.0
    return df['profit'].sum() / len(df) * 100


def quarter(month):
    if month in [8,9,10]:   return 'Q1 (Aug-Oct)'
    if month in [11,12,1]:  return 'Q2 (Nov-Jan)'
    if month in [2,3]:      return 'Q3 (Feb-Mar)'
    return 'Q4 (Apr-May)'


def odds_range(o):
    if o < 1.5:   return '1.0-1.5'
    if o < 2.0:   return '1.5-2.0'
    if o < 2.5:   return '2.0-2.5'
    if o < 3.0:   return '2.5-3.0'
    if o < 4.0:   return '3.0-4.0'
    if o < 6.0:   return '4.0-6.0'
    return '6.0+'


# ══════════════════════════════════════════════════════════════════════════════
# STAP 8 — Run simulaties
# ══════════════════════════════════════════════════════════════════════════════
res_ekf   = simuleer(merged,
                     'ekf_prob_home',   'ekf_prob_draw',   'ekf_prob_away',
                     'bfe_prob_home',   'bfe_prob_draw',   'bfe_prob_away',
                     'BFEH','BFED','BFEA', EDGE)

res_naive = simuleer(merged,
                     'naive_prob_home', 'naive_prob_draw', 'naive_prob_away',
                     'bfe_prob_home',   'bfe_prob_draw',   'bfe_prob_away',
                     'BFEH','BFED','BFEA', EDGE)

# ══════════════════════════════════════════════════════════════════════════════
# TABLE A — ROI per drempel
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("TABLE A: ROI per edge threshold")
print("="*65)
print(f"{'Threshold':>10} | {'EKF bets':>9} {'EKF ROI':>9} | {'Naive bets':>10} {'Naive ROI':>10}")
print("-"*65)

for drempel in [0.00, 0.02, 0.05, 0.08, 0.10, 0.15, 0.20]:
    e = simuleer(merged,
                 'ekf_prob_home','ekf_prob_draw','ekf_prob_away',
                 'bfe_prob_home','bfe_prob_draw','bfe_prob_away',
                 'BFEH','BFED','BFEA', drempel)
    n = simuleer(merged,
                 'naive_prob_home','naive_prob_draw','naive_prob_away',
                 'bfe_prob_home','bfe_prob_draw','bfe_prob_away',
                 'BFEH','BFED','BFEA', drempel)
    print(f"{drempel:>10.2f} | {len(e):>9} {roi(e):>+8.1f}% | {len(n):>10} {roi(n):>+9.1f}%")

# ══════════════════════════════════════════════════════════════════════════════
# TABLE B — ROI per uitkomsttype bij 5% drempel
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*65)
print(f"TABLE B: Betting results")
print("="*65)
print(f"{'Model':<12} {'Outcome':<8} {'Bets':>6} {'Profit':>8} {'ROI':>8}")
print("-"*65)

for label, res in [('EKF', res_ekf), ('Naive', res_naive)]:
    for outcome, name in [('Overall','Overall'), ('H','Home'), ('D','Draw'), ('A','Away')]:
        sub = res if outcome == 'Overall' else res[res['bet'] == outcome]
        profit = sub['profit'].sum() if len(sub) > 0 else 0
        print(f"{label:<12} {name:<8} {len(sub):>6} {profit:>+8.2f} {roi(sub):>+7.1f}%")

# ══════════════════════════════════════════════════════════════════════════════
# TABLE C — ROI per kwartaal
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*65)
print(f"TABLE C: ROI per quarter")
print("="*65)
print(f"{'Quarter':<14} {'EKF bets':>9} {'EKF ROI':>9} | {'Naive bets':>10} {'Naive ROI':>10}")
print("-"*65)

for label, res in [('EKF', res_ekf), ('Naive', res_naive)]:
    res['quarter'] = res['month'].apply(quarter)

quarters = ['Q1 (Aug-Oct)','Q2 (Nov-Jan)','Q3 (Feb-Mar)','Q4 (Apr-May)']
for q in quarters:
    e_q = res_ekf[res_ekf['quarter'] == q]
    n_q = res_naive[res_naive['quarter'] == q]
    print(f"{q:<14} {len(e_q):>9} {roi(e_q):>+8.1f}% | {len(n_q):>10} {roi(n_q):>+9.1f}%")

# ══════════════════════════════════════════════════════════════════════════════
# TABLE D — ROI per odds range
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*65)
print(f"TABLE D: ROI per odds range")
print("="*65)
print(f"{'Odds range':<12} {'Bets':>6} {'Win rate':>9} {'Profit':>8} {'ROI':>8}")
print("-"*65)

res_ekf['odds_range'] = res_ekf['odds'].apply(odds_range)
for r in ['1.0-1.5','1.5-2.0','2.0-2.5','2.5-3.0','3.0-4.0','4.0-6.0','6.0+']:
    sub = res_ekf[res_ekf['odds_range'] == r]
    wr  = sub['won'].mean()*100 if len(sub) > 0 else 0
    profit = sub['profit'].sum() if len(sub) > 0 else 0
    print(f"{r:<12} {len(sub):>6} {wr:>8.1f}% {profit:>+8.2f} {roi(sub):>+7.1f}%")

# ══════════════════════════════════════════════════════════════════════════════
# TABLE E — Top/bottom teams (EKF)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*65)
print(f"TABLE E: ROI per team")
print("="*65)

team_rows = []
for team in sorted(set(merged['home_team'].unique()) | set(merged['away_team'].unique())):
    sub = res_ekf[(res_ekf['home'] == team) | (res_ekf['away'] == team)]
    if len(sub) >= 5:
        team_rows.append({'team': team, 'bets': len(sub),
                          'profit': sub['profit'].sum(), 'roi': roi(sub)})

team_df = pd.DataFrame(team_rows).sort_values('roi', ascending=False)
print("\nTop 5:")
print(f"{'Team':<28} {'Bets':>6} {'Profit':>8} {'ROI':>8}")
print("-"*55)
for _, r in team_df.head(5).iterrows():
    print(f"{r['team']:<28} {int(r['bets']):>6} {r['profit']:>+8.2f} {r['roi']:>+7.1f}%")

print("\nBottom 5:")
print(f"{'Team':<28} {'Bets':>6} {'Profit':>8} {'ROI':>8}")
print("-"*55)
for _, r in team_df.tail(5).iterrows():
    print(f"{r['team']:<28} {int(r['bets']):>6} {r['profit']:>+8.2f} {r['roi']:>+7.1f}%")

# ══════════════════════════════════════════════════════════════════════════════
# STAP 9 — Random baseline (1000x simulaties)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("RANDOM BASELINE (1000x simulation)")
print("="*65)

random.seed(RANDOM_SEED)
sim_base = merged[merged['BFEH'].notna()].copy()
random_rois = []

for _ in range(N_RANDOM):
    winst = bets = 0
    for _, row in sim_base.iterrows():
        edges = {
            'H': random.uniform(0,1) - row['bfe_prob_home'],
            'D': random.uniform(0,1) - row['bfe_prob_draw'],
            'A': random.uniform(0,1) - row['bfe_prob_away'],
        }
        best = max(edges, key=edges.get)
        if edges[best] >= 0.0:
            bets += 1
            o = row['BFEH' if best=='H' else 'BFED' if best=='D' else 'BFEA']
            if pd.isna(o) or o <= 1:
                continue
            winst += (o - 1) if row['result'] == best else -1
    if bets > 0:
        random_rois.append(winst / bets * 100)

p95     = np.percentile(random_rois, 95)
res_ekf_0 = simuleer(merged,
                     'ekf_prob_home','ekf_prob_draw','ekf_prob_away',
                     'bfe_prob_home','bfe_prob_draw','bfe_prob_away',
                     'BFEH','BFED','BFEA', 0.0)
ekf_roi = roi(res_ekf_0)

# Exacte p-waarde: fractie simulaties met ROI >= EKF ROI
p_value = np.mean(np.array(random_rois) >= ekf_roi)

print(f"Random mean ROI:   {np.mean(random_rois):+.1f}%")
print(f"Random 95th pct:   {p95:+.1f}%")
print(f"EKF ROI:           {ekf_roi:+.1f}%")
print(f"Exact p-value:     {p_value:.4f}")
print(f"Significant (p<.05): {'Yes' if p_value < 0.05 else 'No'}")

# Opslaan
res_ekf.to_csv(os.path.join(BASE_DIR, "betting_results_ekf_bfe.csv"), index=False)
res_naive.to_csv(os.path.join(BASE_DIR, "betting_results_naive_bfe.csv"), index=False)
print("\nResultaten opgeslagen.")

Finale dataset: 654 wedstrijden

TABLE A: ROI per edge threshold
 Threshold |  EKF bets   EKF ROI | Naive bets  Naive ROI
-----------------------------------------------------------------
      0.00 |       654    +19.1% |        654      -0.7%
      0.02 |       525    +21.0% |        650      -0.9%
      0.05 |       316    +15.5% |        583      -1.4%
      0.08 |       159     -6.0% |        491      -5.2%
      0.10 |        98     -4.0% |        424      -7.9%
      0.15 |        21    +45.1% |        319      -6.4%
      0.20 |         1   -100.0% |        195     -28.3%

TABLE B: Betting results
Model        Outcome    Bets   Profit      ROI
-----------------------------------------------------------------
EKF          Overall     654  +125.03   +19.1%
EKF          Home        288    +5.05    +1.8%
EKF          Draw        155   +67.20   +43.4%
EKF          Away        211   +52.78   +25.0%
Naive        Overall     654    -4.35    -0.7%
Naive        Home        301   -40.25  